# Purpose:
- Gather information about GCaMP7 pan-inhibitory data (to compare with GCaMP8)
- Make csv files to upload and trigger

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import os

from brain_observatory_qc.data_access import from_lims

from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorOphysProjectCache as bpc

from pymongo import MongoClient
mongo = MongoClient("flaskapp.corp.alleninstitute.org", 27017)

cache = bpc.from_lims()
table = cache.get_ophys_experiment_table(passed_only=False)

In [2]:
# helper function
def get_session_info_per_group_of_mice(mouse_ids, gcamp_type):
    experiment_table = pd.DataFrame()
    problem_experiments = []
    for mouse_id in mouse_ids:
        mouse_data = from_lims.get_imaging_ids_for_mouse_id(mouse_id)
        if len(mouse_data) > 0:
            ophys_experiment_ids = mouse_data.ophys_experiment_id.values
            for ophys_experiment_id in ophys_experiment_ids: 
                try: 
                    expt_info = from_lims.get_general_info_for_ophys_experiment_id(ophys_experiment_id)
                    if len(expt_info) > 0:
                        expt_info = expt_info.iloc[[0]]
                        genotype = from_lims.get_genotype_for_ophys_experiment_id(ophys_experiment_id)
                        expt_info["mouse_id"] = mouse_id # expt_info only contains `donor_id` and `specimen_id`, need to add mouse_id here
                        expt_info["full_genotype"] = genotype.full_genotype.values[0] # base function doesnt get genotype, add it
                        experiment_table = pd.concat([experiment_table, expt_info])
                except: 
                    # print('problem for mouse_id: ', mouse_id, ', expt_id: ', ophys_experiment_id)
                    problem_experiments.append(ophys_experiment_id)
    experiment_table['gcamp'] = gcamp_type
    return experiment_table, problem_experiments


def get_session_info_per_group_of_sessions(session_ids, mouse_ids, gcamp_type):
    experiment_table = pd.DataFrame()
    problem_experiments = []
    for session_id, mouse_id in zip(session_ids, mouse_ids):
        ophys_experiment_ids = from_lims.get_ophys_experiment_ids_for_ophys_session_id(session_id).ophys_experiment_id.values
        for ophys_experiment_id in ophys_experiment_ids: 
            try: 
                expt_info = from_lims.get_general_info_for_ophys_experiment_id(ophys_experiment_id)
                if len(expt_info) > 0:
                    expt_info = expt_info.iloc[[0]]
                    genotype = from_lims.get_genotype_for_ophys_experiment_id(ophys_experiment_id)
                    expt_info["mouse_id"] = mouse_id # expt_info only contains `donor_id` and `specimen_id`, need to add mouse_id here
                    expt_info["full_genotype"] = genotype.full_genotype.values[0] # base function doesnt get genotype, add it
                    experiment_table = pd.concat([experiment_table, expt_info])
            except: 
                # print('problem for mouse_id: ', mouse_id, ', expt_id: ', ophys_experiment_id)
                problem_experiments.append(ophys_experiment_id)
    experiment_table['gcamp'] = gcamp_type
    return experiment_table, problem_experiments


def get_mongo_client(username, password, host, port):
    client = MongoClient(f'mongodb://{username}:{password}@{host}:{port}')
    return client


def get_zdrift(opid):
    try:
        client = get_mongo_client('public', 'public_password', 'qc-sys-db', 27017)
    except:
        raise('Failed to connect to mongo')
    
    opid = int(opid)
    try:
        record = client.records.metrics.find_one({'data_id': opid})
    except:
        raise('No record found for opid: ', opid)
        
    try:
        zdrift = record['local_z_stack']['z_drift_corr_um_diff']
    except:
        zdrift = np.nan
    return zdrift

def get_intensity_drift(opid):
    try:
        client = get_mongo_client('public', 'public_password', 'qc-sys-db', 27017)
    except:
        raise('Failed to connect to mongo')
    
    opid = int(opid)
    try:
        record = client.records.metrics.find_one({'data_id': opid})
    except:
        raise('No record found for opid: ', opid)
        
    try:
        intensity_drift = record['motion_corr_physio']['percent_change_intensity']
    except:
        intensity_drift = np.nan
    return intensity_drift


def get_monitor_sync(osid):
    try:
        client = get_mongo_client('public', 'public_password', 'qc-sys-db', 27017)
    except:
        raise('Failed to connect to mongo')
    
    osid = int(osid)
    try:
        record = client.records.metrics.find_one({'data_id': osid})
    except:
        raise('No record found for osid: ', osid)
        
    try:
        monitor_sync = record['rig_sync']['display_lag']
    except:
        monitor_sync = np.nan
    return monitor_sync